# Object Tracking Basics

This notebook introduces a minimal object-tracking workflow using detections from consecutive image frames.

The objective is to associate detections across frames and assign a persistent track identifier to the same object over time.

In [1]:
# 1-10D — Environment Check

import numpy as np
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)
print("Object tracking environment ready.")

NumPy version: 2.4.6
Object tracking environment ready.


## Detection Representation

Each detected object is represented by a bounding box:

`[x_min, y_min, x_max, y_max]`

A detection also contains a class label and confidence score.

Tracking adds a persistent track ID so that the same object can be followed across consecutive frames.

In [2]:
# 2-10D — Example Detections

detections_frame_1 = [
    {"bbox": [100, 120, 220, 300], "class": "car", "score": 0.91},
    {"bbox": [400, 150, 470, 280], "class": "person", "score": 0.87},
]

detections_frame_2 = [
    {"bbox": [112, 125, 232, 305], "class": "car", "score": 0.89},
    {"bbox": [405, 153, 475, 283], "class": "person", "score": 0.85},
]

print("Frame 1:", detections_frame_1)
print("Frame 2:", detections_frame_2)

Frame 1: [{'bbox': [100, 120, 220, 300], 'class': 'car', 'score': 0.91}, {'bbox': [400, 150, 470, 280], 'class': 'person', 'score': 0.87}]
Frame 2: [{'bbox': [112, 125, 232, 305], 'class': 'car', 'score': 0.89}, {'bbox': [405, 153, 475, 283], 'class': 'person', 'score': 0.85}]


## Bounding-Box Centre

A simple way to compare object locations between frames is to calculate the centre of each bounding box.

For a bounding box `[x_min, y_min, x_max, y_max]`:

`x_centre = (x_min + x_max) / 2`

`y_centre = (y_min + y_max) / 2`

In [3]:
# 3-10D — Calculate Bounding-Box Centres

def bbox_center(bbox):
    x_min, y_min, x_max, y_max = bbox

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2

    return np.array([x_center, y_center])


for detection in detections_frame_1:
    detection["center"] = bbox_center(detection["bbox"])

for detection in detections_frame_2:
    detection["center"] = bbox_center(detection["bbox"])

print("Frame 1 centres:")
for detection in detections_frame_1:
    print(detection["class"], detection["center"])

print("\nFrame 2 centres:")
for detection in detections_frame_2:
    print(detection["class"], detection["center"])

Frame 1 centres:
car [160. 210.]
person [435. 215.]

Frame 2 centres:
car [172. 215.]
person [440. 218.]


## Nearest-Centre Association

A minimal tracking approach can associate detections between consecutive frames by comparing their bounding-box centres.

The detection in the new frame with the smallest centre distance is considered the corresponding object when the distance is below a selected association threshold.

In [4]:
# 4-10D — Centre-Distance Association

association_threshold = 50.0

for current in detections_frame_2:
    best_match = None
    best_distance = float("inf")

    for previous in detections_frame_1:
        distance = np.linalg.norm(
            current["center"] - previous["center"]
        )

        if distance < best_distance:
            best_distance = distance
            best_match = previous

    if best_distance <= association_threshold:
        current["track_id"] = best_match["class"]
    else:
        current["track_id"] = "new"

    print(
        f"{current['class']} -> "
        f"track: {current['track_id']}, "
        f"distance: {best_distance:.2f} px"
    )

car -> track: car, distance: 13.00 px
person -> track: person, distance: 5.83 px


## Persistent Track IDs

In a real tracker, the track identifier should be independent of the object class.

The following example assigns numerical track IDs and demonstrates the basic idea of maintaining object identity across frames.

In [5]:
# 5-10D — Assign Track IDs

tracks = {
    1: {"class": "car", "center": detections_frame_1[0]["center"]},
    2: {"class": "person", "center": detections_frame_1[1]["center"]},
}

for current in detections_frame_2:
    best_track_id = None
    best_distance = float("inf")

    for track_id, track in tracks.items():
        distance = np.linalg.norm(
            current["center"] - track["center"]
        )

        if distance < best_distance:
            best_distance = distance
            best_track_id = track_id

    if best_distance <= association_threshold:
        current["track_id"] = best_track_id
        tracks[best_track_id]["center"] = current["center"]
    else:
        new_id = max(tracks.keys()) + 1
        current["track_id"] = new_id
        tracks[new_id] = {
            "class": current["class"],
            "center": current["center"]
        }

for detection in detections_frame_2:
    print(
        f"Track ID {detection['track_id']}: "
        f"{detection['class']}, "
        f"centre={detection['center']}"
    )

Track ID 1: car, centre=[172. 215.]
Track ID 2: person, centre=[440. 218.]


## Tracking Across Multiple Frames

The following experiment simulates the motion of a vehicle across several consecutive frames.

The object moves approximately 10 pixels to the right in each frame.

In [6]:
# 6-10D — Simulated Multi-Frame Tracking

frames = [
    [110, 120, 230, 300],
    [120, 120, 240, 300],
    [130, 121, 250, 301],
    [140, 122, 260, 302],
]

track_id = 1

previous_center = bbox_center(frames[0])

print(f"Frame 1: Track ID {track_id}, centre {previous_center}")

for frame_index, bbox in enumerate(frames[1:], start=2):
    current_center = bbox_center(bbox)

    distance = np.linalg.norm(current_center - previous_center)

    if distance <= association_threshold:
        print(
            f"Frame {frame_index}: "
            f"Track ID {track_id}, "
            f"centre {current_center}, "
            f"movement {distance:.2f} px"
        )
    else:
        print(
            f"Frame {frame_index}: "
            f"new track required"
        )

    previous_center = current_center

Frame 1: Track ID 1, centre [170. 210.]
Frame 2: Track ID 1, centre [180. 210.], movement 10.00 px
Frame 3: Track ID 1, centre [190. 211.], movement 10.05 px
Frame 4: Track ID 1, centre [200. 212.], movement 10.05 px


## Tracking Limitation

Nearest-centre association is a minimal tracking method.

It can work for simple cases with slow object motion and well-separated objects, but it can fail when objects cross, disappear, move rapidly, or become occluded.

More advanced tracking systems combine detection, motion prediction, data association, and track management.

In [7]:
# 7-10D — Tracking Summary

print("Object tracking experiment complete.")
print()
print("The experiment demonstrated:")
print("- Detection representation using bounding boxes")
print("- Bounding-box centre calculation")
print("- Detection association between frames")
print("- Persistent track IDs")
print("- Basic multi-frame tracking")
print()
print("The simple nearest-centre method is suitable as a conceptual")
print("baseline but is not sufficient for robust autonomous-driving tracking.")

Object tracking experiment complete.

The experiment demonstrated:
- Detection representation using bounding boxes
- Bounding-box centre calculation
- Detection association between frames
- Persistent track IDs
- Basic multi-frame tracking

The simple nearest-centre method is suitable as a conceptual
baseline but is not sufficient for robust autonomous-driving tracking.


## Object Tracking — Conclusion

The experiment demonstrated how detections from consecutive frames can be associated to maintain object identity over time.

A simple nearest-centre association method was sufficient to demonstrate persistent track IDs for objects with small frame-to-frame movement.

The experiment also showed why practical autonomous-driving tracking requires more robust methods that account for object motion, occlusion, missed detections, and ambiguous associations.